# IDH1–PRKDC LLM Explanation (LLM_module)

Runs the in-repo `LLM_module` on a single gene pair (IDH1–PRKDC) and writes a reference-style text report that includes:
- Final explanation text
- CoVe intermediate questions + answers
- Self-refine reviewer feedback

Output directory: `playground/output/IDH1_PRKDC_notebook/`

In [4]:
from __future__ import annotations

import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
SRC = ROOT / "src"
for p in (str(SRC), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

PAIR_A = "IDH1"
PAIR_B = "PRKDC"
TARGET_CONTEXT = "Glioma"

OUT_DIR = ROOT / "playground" / "output" / f"{PAIR_A}_{PAIR_B}_notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)

ROOT: /home/guoyu/grad_design/KG-LLM-XSL
OUT_DIR: /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook


In [5]:
# Load the prompt exported by algorithm_module (preferred), otherwise rebuild from the saved subgraph graphml.

import networkx as nx

from algorithm_module.graph_search_algo import gene_node
from algorithm_module.utils.graph_search_utils import load_graphml

PROMPT_ROOT = Path("/data/guoyu/KG-LLM-XSL/output/gene_pairs_subgraphs")
pair_dir = PROMPT_ROOT / f"{PAIR_A}_{PAIR_B}"
prompt_txt_path = pair_dir / f"{PAIR_A}_{PAIR_B}_prompt.txt"
prompt_msg_path = pair_dir / f"{PAIR_A}_{PAIR_B}_prompt_messages.json"

system_prompt: str | None = None
user_prompt: str | None = None
prompt_source: str | None = None

if prompt_msg_path.exists():
    obj = json.loads(prompt_msg_path.read_text(encoding="utf-8"))
    msgs = obj.get("messages", []) if isinstance(obj, dict) else []
    if isinstance(msgs, list):
        for m in msgs:
            if not isinstance(m, dict):
                continue
            role = str(m.get("role", "")).strip().lower()
            content = m.get("content")
            if not isinstance(content, str):
                continue
            if role == "system" and system_prompt is None:
                system_prompt = content
            elif role == "user" and user_prompt is None:
                user_prompt = content
    prompt_source = str(prompt_msg_path)
elif prompt_txt_path.exists():
    user_prompt = prompt_txt_path.read_text(encoding="utf-8")
    prompt_source = str(prompt_txt_path)

if not user_prompt:
    from algorithm_module.subgraph_extraction import build_chat_prompts, export_llm_csv

    sub_graphml_candidates = [
        OUT_DIR / f"{PAIR_A}_{PAIR_B}_subgraph.graphml",
        OUT_DIR / "subgraph.graphml",
    ]
    sub_path = next((p for p in sub_graphml_candidates if p.exists()), None)
    if sub_path is None:
        raise FileNotFoundError(
            "No prompt found under /data/... and no subgraph GraphML found in OUT_DIR. "
            "Run the subgraph extraction notebook first to produce *_subgraph.graphml."
        )

    sub = load_graphml(sub_path)
    core_a, core_b = gene_node(PAIR_A), gene_node(PAIR_B)
    cores = {core_a, core_b}

    path_nodes: set[str] = set()
    und = sub.to_undirected(as_view=True)
    if core_a in und and core_b in und:

        for p in nx.all_shortest_paths(und, core_a, core_b):
            path_nodes.update(p)


    node_rows, edge_rows = export_llm_csv(sub, cores=cores, path_nodes=path_nodes, out_dir=OUT_DIR)
    system_prompt, user_prompt = build_chat_prompts(PAIR_A, PAIR_B, node_rows=node_rows, edge_rows=edge_rows)
    prompt_source = str(sub_path)

print("Prompt source:", prompt_source)
print("Has system prompt:", system_prompt is not None)
print("User prompt chars:", len(user_prompt or ""))

Prompt source: /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/IDH1_PRKDC_subgraph.graphml
Has system prompt: True
User prompt chars: 9691


In [6]:
# Run CoVe + self-refine (stock LLM_module prompts) and write a reference-style report (.txt).

from LLM_module import eval_config as ecfg
from LLM_module.utils.llm_client import get_default_client
from LLM_module.utils import llm_strategies as strat

client = get_default_client()

# Make multi-call strategies more robust on flaky networks (does not change prompts)
ecfg.LLM_MAX_RETRY = max(int(getattr(ecfg, "LLM_MAX_RETRY", 3)), 6)
ecfg.LLM_REQUEST_TIMEOUT_S = max(float(getattr(ecfg, "LLM_REQUEST_TIMEOUT_S", 60.0)), 120.0)

print("BASELINE_CONFIG:", getattr(ecfg, "BASELINE_CONFIG", None))
print("LLM_PROVIDER:", getattr(ecfg, "LLM_PROVIDER", None))
print("MODEL:", getattr(ecfg, "MODEL", None))

# 1) CoVe (use original prompt; no augmentation)
cove_trace = strat.run_cove(
    client,
    user_prompt or "",
    system_prompt=system_prompt,
    n_questions=3,
    temperature=ecfg.TEMPERATURE,
    top_p=ecfg.TOP_P,
    max_tokens=ecfg.MAX_TOKENS,
 )
cove_final = cove_trace.final.text if cove_trace.final else cove_trace.initial.text

# 2) Self-refine (use stock reviewer/rewrite prompts from LLM_module)
sr_trace = strat.run_self_refine(
    client,
    cove_final,
    system_prompt=system_prompt,
    rounds=1,
    temperature=ecfg.TEMPERATURE,
    top_p=ecfg.TOP_P,
    max_tokens=ecfg.MAX_TOKENS,
 )
final_text = sr_trace.final.text if sr_trace.final else sr_trace.initial.text

out_path = OUT_DIR / f"{PAIR_A}-{PAIR_B}_{TARGET_CONTEXT.lower()}_case_study.txt"

report_lines: list[str] = []
report_lines.append(f"PAIR: {PAIR_A}-{PAIR_B}")
report_lines.append(f"TARGET_CONTEXT: {TARGET_CONTEXT}")
report_lines.append("")
report_lines.append("=== GENERATED (LLM_Module; CoVe + self-refine) ===")
report_lines.append((final_text or "").strip())
report_lines.append("")
report_lines.append("=== TRACE (for debugging) ===")
report_lines.append("[Prompt source]")
report_lines.append(str(prompt_source))
report_lines.append("")
report_lines.append("[Model]")
report_lines.append(str(getattr(ecfg, "MODEL", "")))
report_lines.append("")
report_lines.append("[CoVe questions]")
report_lines.append((cove_trace.questions.text if cove_trace.questions else "").strip())
report_lines.append("")
report_lines.append("[CoVe answers]")
report_lines.append((cove_trace.answers.text if cove_trace.answers else "").strip())
report_lines.append("")
report_lines.append("[Self-refine feedback]")
report_lines.append((sr_trace.feedback.text if sr_trace.feedback else "").strip())
report_lines.append("")
report_lines.append("NOTE: single-pair run; no metric scoring computed here.")

out_path.write_text("\n".join(report_lines).rstrip() + "\n", encoding="utf-8")
print("Wrote:", out_path)
print("Final chars:", len(final_text or ""))

BASELINE_CONFIG: gpt-3.5-turbo_config.json
LLM_PROVIDER: aigcbest
MODEL: openai/gpt-3.5-turbo
Wrote: /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/IDH1-PRKDC_glioma_case_study.txt
Final chars: 10366


In [7]:
# Preview first ~60 lines of the generated report
preview = out_path.read_text(encoding="utf-8").splitlines()
print("Lines:", len(preview))
print("\n".join(preview[:60]))

Lines: 141
PAIR: IDH1-PRKDC
TARGET_CONTEXT: Glioma

=== GENERATED (LLM_Module; CoVe + self-refine) ===
1) Mechanism Name: Dual Stress Vulnerability via Impaired Metabolic Redox Homeostasis and Defective DNA Repair

2) Mechanistic Summary:  
The synthetic lethality (SL) between IDH1 and PRKDC likely arises from two orthogonal yet synergistic vulnerabilities. The primary functional aspect involves IDH1’s role in metabolic pathways, specifically NADPH regeneration, cellular response to chemical stress, and the KEAP1-NFE2L2 antioxidant pathway, as supported by IDH1’s membership in pathways R-HSA-389542, R-HSA-9711123, and R-HSA-9755511 (gene:IDH1 -> pathway:R-HSA-389542, R-HSA-9711123, R-HSA-9755511 | Reactome). This suggests that IDH1 loss or mutation impairs redox homeostasis and antioxidant defenses, elevating oxidative stress. The secondary, orthogonal functional aspect involves PRKDC’s central role in DNA double-strand break repair through the Nonhomologous End-Joining (NHEJ) pathway,